# web search resulty by using lanchain built in duckduckgo

In [1]:
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper


wrapper = DuckDuckGoSearchAPIWrapper(
    max_results=50,
    region="in-en",       
    time="y"              
)


search_results = DuckDuckGoSearchResults(
    api_wrapper=wrapper,
    output_format="list",
    num_results=10
)

while True:
    query = input("\nWhat do you want to search (or type 'exit'): ").strip()
    if query.lower() == "exit":
        break

    if not query:
        continue

    print(f"\n--- Results for: {query} ---")
    results = search_results.invoke(query)

    if isinstance(results, list):
        for index, item in enumerate(results, start=1):
            title = item.get("title", "No Title")
            link = item.get("link", "No Link")
            snippet = item.get("snippet", "No Snippet")

            print(f"[{index}] {title}")
            print(f"    Link:    {link}")
            print(f"    Snippet: {snippet}\n")
    else:
     
        print(results)

ModuleNotFoundError: No module named 'langchain_community'

In [19]:
from langchain_ollama import ChatOllama
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
import requests
from bs4 import BeautifulSoup

# Initialize local Ollama model (Make sure Ollama app is running locally)
# Options: "llama3", "mistral", "gemma2", "phi3"
llm = ChatOllama(model="llama3", temperature=0.3)
print("✅ Ollama LLM Initialized!")


✅ Ollama LLM Initialized!


In [20]:
# Initialize DuckDuckGo Search Tool
wrapper = DuckDuckGoSearchAPIWrapper()
print("✅ DuckDuckGo Search Tool Ready!")


✅ DuckDuckGo Search Tool Ready!


In [21]:
def fetch_rich_webpage(url):
    """Scrapes full text paragraphs from a webpage URL to get rich data."""
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=5)
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Collect paragraphs over 30 characters
        paragraphs = [p.get_text().strip() for p in soup.find_all("p") if len(p.get_text().strip()) > 30]
        return " ".join(paragraphs[:8])  # Return top 8 paragraphs of rich text
    except Exception as e:
        return f"Could not scrape webpage: {e}"


In [ ]:
from langchain_ollama import ChatOllama
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

# Initialize local model (llama3.2)
llm = ChatOllama(model="llama3.2", temperature=0.3)
wrapper = DuckDuckGoSearchAPIWrapper()

def ollama_search_assistant(query):
    print(f"[1/2] Searching web for: '{query}'...")
    
    try:
        search_results = wrapper.results(query, max_results=3)
    except Exception as e:
        return f"Web search timeout/connection issue: {e}"
        
    if not search_results:
        return "No search results found."
    
    context = "\n".join([f"- {item['title']}: {item['snippet']}" for item in search_results])
    
    print("[2/2] Local Llama 3.2 is analyzing and generating response...\n")
    
    prompt = f"""You are a helpful AI search assistant.
    Use the following web search data to answer the query accurately and concisely.

    Query: {query}
    
    Web Search Context:
    {context}
    """
    
    response = llm.invoke(prompt)
    return response.content

# --- Execute Query ---
answer = ollama_search_assistant("who is pappu")
print("="*60)
print(answer)
print("="*60)


[1/2] Searching web for: 'who is pappu'...
[2/2] Local Llama 3.2 is analyzing and generating response...

